In [1]:
import pandas as pd, numpy as np, geopandas as gpd
import matplotlib.pyplot as plt, os, sys
#import plotly.graph_objects as go
import networkx as nx, math, glob #, fiona
import pickle
from pyvis.network import Network
from shapely import geometry, ops
from shapely.geometry import Point, LineString, MultiLineString

import torch
from torch_geometric.data import Data
from torch_geometric.utils.convert import from_networkx, to_networkx

pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings('ignore')

In [2]:
p = '/Volumes/arl/Eugene/'

In [3]:
gdb_path = p + 'Graph Neural Networks/GNN/GNN.gdb'

In [4]:
df = gpd.read_file(gdb_path, layer = 'AllRds_sub_Overlay_tf_update')
df = df[df.Shape_Length != 0] #remove zero length events from overlay file.
print(len(df))
df.head()

404214


,uid,RT_UNIQUE,BEGIN_MP,END_MP,lrsid,TYPE_OP,RD_NAME,CO_NAME,ADTSTATN,LASTCNT,LASTCNTYR,ADTPRIOR,ADTSINGLE,ADTCOMBO,Shape_Length,geometry
0,1180,001-CR-1001 -000,0.000,0.027,001-CR-1001 -000,2.0,TAYLOR FORD RD,Adair,0,0,0,0,0,0,125.690261,"MULTILINESTRING ((5083050.500 3562982.600, 508..."
1,1179,001-CR-1215 -000,0.000,0.498,001-CR-1215 -000,2.0,J SAMUELL RD,Adair,0,0,0,0,0,0,2629.928650,"MULTILINESTRING ((5060128.900 3524658.500, 506..."
2,1177,001-CR-1173 -000,0.757,0.898,001-CR-1173 -000,2.0,RUTLEDGE RD,Adair,0,0,0,0,0,0,745.439675,"MULTILINESTRING ((5092902.500 3548903.000, 509..."
3,1176,001-CR-1070 -000,0.000,1.171,001-CR-1070 -000,2.0,REDDIE TUCKER RD,Adair,0,0,0,0,0,0,6183.250593,"MULTILINESTRING ((5078102.100 3595321.300, 507..."
4,1167,001-CR-1001 -000,0.027,0.659,001-CR-1001 -000,2.0,TAYLOR FORD RD,Adair,0,0,0,0,0,0,3353.470541,"MULTILINESTRING ((5082945.600 3563049.400, 508..."


In [5]:
df['Truck_AADT'] = df['ADTSINGLE'] + df['ADTCOMBO']

In [6]:
df['Length'] = abs(df['END_MP'] - df['BEGIN_MP'])
df.head()

,uid,RT_UNIQUE,BEGIN_MP,END_MP,lrsid,TYPE_OP,RD_NAME,CO_NAME,ADTSTATN,LASTCNT,LASTCNTYR,ADTPRIOR,ADTSINGLE,ADTCOMBO,Shape_Length,geometry,Truck_AADT,Length
0,1180,001-CR-1001 -000,0.000,0.027,001-CR-1001 -000,2.0,TAYLOR FORD RD,Adair,0,0,0,0,0,0,125.690261,"MULTILINESTRING ((5083050.500 3562982.600, 508...",0,0.027
1,1179,001-CR-1215 -000,0.000,0.498,001-CR-1215 -000,2.0,J SAMUELL RD,Adair,0,0,0,0,0,0,2629.928650,"MULTILINESTRING ((5060128.900 3524658.500, 506...",0,0.498
2,1177,001-CR-1173 -000,0.757,0.898,001-CR-1173 -000,2.0,RUTLEDGE RD,Adair,0,0,0,0,0,0,745.439675,"MULTILINESTRING ((5092902.500 3548903.000, 509...",0,0.141
3,1176,001-CR-1070 -000,0.000,1.171,001-CR-1070 -000,2.0,REDDIE TUCKER RD,Adair,0,0,0,0,0,0,6183.250593,"MULTILINESTRING ((5078102.100 3595321.300, 507...",0,1.171
4,1167,001-CR-1001 -000,0.027,0.659,001-CR-1001 -000,2.0,TAYLOR FORD RD,Adair,0,0,0,0,0,0,3353.470541,"MULTILINESTRING ((5082945.600 3563049.400, 508...",0,0.632


In [7]:
df[df.Length == 0]

,uid,RT_UNIQUE,BEGIN_MP,END_MP,lrsid,TYPE_OP,RD_NAME,CO_NAME,ADTSTATN,LASTCNT,LASTCNTYR,ADTPRIOR,ADTSINGLE,ADTCOMBO,Shape_Length,geometry,Truck_AADT,Length


In [8]:
df.geom_type.unique()

array(['MultiLineString'], dtype=object)

In [9]:
def merge_lines(x):
    return ops.linemerge(x)

In [10]:
def latlong(x):
    return round(x.coords.xy[1][0],6), round(x.coords.xy[0][0], 6), round(x.coords.xy[1][-1], 6), round(x.coords.xy[0][-1], 6)

In [11]:
df['geometry'] = df['geometry'].apply(merge_lines)
print(df.geom_type.unique())
print(len(df))
df.head()

['LineString' 'MultiLineString']
404214


,uid,RT_UNIQUE,BEGIN_MP,END_MP,lrsid,TYPE_OP,RD_NAME,CO_NAME,ADTSTATN,LASTCNT,LASTCNTYR,ADTPRIOR,ADTSINGLE,ADTCOMBO,Shape_Length,geometry,Truck_AADT,Length
0,1180,001-CR-1001 -000,0.000,0.027,001-CR-1001 -000,2.0,TAYLOR FORD RD,Adair,0,0,0,0,0,0,125.690261,"LINESTRING (5083050.500 3562982.600, 5083024.6...",0,0.027
1,1179,001-CR-1215 -000,0.000,0.498,001-CR-1215 -000,2.0,J SAMUELL RD,Adair,0,0,0,0,0,0,2629.928650,"LINESTRING (5060128.900 3524658.500, 5060109.7...",0,0.498
2,1177,001-CR-1173 -000,0.757,0.898,001-CR-1173 -000,2.0,RUTLEDGE RD,Adair,0,0,0,0,0,0,745.439675,"LINESTRING (5092902.500 3548903.000, 5092826.0...",0,0.141
3,1176,001-CR-1070 -000,0.000,1.171,001-CR-1070 -000,2.0,REDDIE TUCKER RD,Adair,0,0,0,0,0,0,6183.250593,"LINESTRING (5078102.100 3595321.300, 5078048.8...",0,1.171
4,1167,001-CR-1001 -000,0.027,0.659,001-CR-1001 -000,2.0,TAYLOR FORD RD,Adair,0,0,0,0,0,0,3353.470541,"LINESTRING (5082945.600 3563049.400, 5082926.3...",0,0.632


In [12]:
dfm = df[df.geom_type == 'MultiLineString']
#dfm.to_csv('check.csv')
dfm.head()

,uid,RT_UNIQUE,BEGIN_MP,END_MP,lrsid,TYPE_OP,RD_NAME,CO_NAME,ADTSTATN,LASTCNT,LASTCNTYR,ADTPRIOR,ADTSINGLE,ADTCOMBO,Shape_Length,geometry,Truck_AADT,Length
8740,19755,004-ST-1210 -000,0.000,0.293,004-ST-1210 -000,2.0,ST-1210,Ballard,0,0,0,0,0,0,1550.940891,"MULTILINESTRING ((3952687.300 3595852.100, 395...",0,0.293
30332,65002,010-CS-2520 -000,0.024,0.095,010-CS-2520 -000,2.0,35TH ST,Boyd,0,0,0,0,0,0,397.141114,"MULTILINESTRING ((5817887.900 4072293.000, 581...",0,0.071
36874,77839,013-CR-1059 -000,0.359,0.937,013-CR-1059 -000,2.0,BETHANY RD,Breathitt,0,0,0,0,0,0,3053.585361,"MULTILINESTRING ((5580570.900 3757846.400, 558...",0,0.578
51620,109752,018-CR-1275G -000,0.086,0.119,018-CR-1275G -000,2.0,OAKWOOD CIR,Calloway,0,0,0,0,0,0,174.398653,"MULTILINESTRING ((4152334.500 3384915.000, 415...",0,0.033
57689,122379,019-CS-1012 -000,0.403,0.433,019-CS-1012 -000,2.0,PARK AVE,Campbell,0,0,0,0,0,0,165.632249,"MULTILINESTRING ((5280572.400 4287089.000, 528...",0,0.030


### Make modifications

In [13]:
uid_19755 = df.loc[df.uid == 19755, 'geometry'].iloc[0]
line_19755 = max(uid_19755.geoms, key=lambda line: line.length)
df.loc[df.uid == 19755, 'geometry'] = line_19755

In [14]:
uid_77839 = df.loc[df.uid == 77839, 'geometry'].iloc[0]
line_77839 = max(uid_77839.geoms, key=lambda line: line.length)
df.loc[df.uid == 77839, 'geometry'] = line_77839

In [15]:
uid_109752 = df.loc[df.uid == 109752, 'geometry'].iloc[0]
line_109752 = max(uid_109752.geoms, key=lambda line: line.length)
df.loc[df.uid == 109752, 'geometry'] = line_109752

In [16]:
uid_180394 = df.loc[df.uid == 180394, 'geometry'].iloc[0]
line_180394 = max(uid_180394.geoms, key=lambda line: line.length)
df.loc[df.uid == 180394, 'geometry'] = line_180394

In [17]:
uid_222438 = df.loc[df.uid == 222438, 'geometry'].iloc[0]
line_222438 = max(uid_222438.geoms, key=lambda line: line.length)
df.loc[df.uid == 222438, 'geometry'] = line_222438

In [18]:
uid_270222 = df.loc[df.uid == 270222, 'geometry'].iloc[0]
line_270222 = max(uid_270222.geoms, key=lambda line: line.length)
df.loc[df.uid == 270222, 'geometry'] = line_270222

In [19]:
uid_305878 = df.loc[df.uid == 305878, 'geometry'].iloc[0]
line_305878 = max(uid_305878.geoms, key=lambda line: line.length)
df.loc[df.uid == 305878, 'geometry'] = line_305878

In [20]:
uid_320202 = df.loc[df.uid == 320202, 'geometry'].iloc[0]
line_320202 = max(uid_320202.geoms, key=lambda line: line.length)
df.loc[df.uid == 320202, 'geometry'] = line_320202

In [21]:
uid_335727 = df.loc[df.uid == 335727, 'geometry'].iloc[0]
line_335727 = max(uid_335727.geoms, key=lambda line: line.length)
df.loc[df.uid == 335727, 'geometry'] = line_335727

In [22]:
uid_355676 = df.loc[df.uid == 355676, 'geometry'].iloc[0]
line_355676 = max(uid_355676.geoms, key=lambda line: line.length)
df.loc[df.uid == 355676, 'geometry'] = line_355676

In [23]:
uid_374006 = df.loc[df.uid == 374006, 'geometry'].iloc[0]
line_374006 = max(uid_374006.geoms, key=lambda line: line.length)
df.loc[df.uid == 374006, 'geometry'] = line_374006

In [24]:
uid_405278 = df.loc[df.uid == 405278, 'geometry'].iloc[0]
line_405278 = max(uid_405278.geoms, key=lambda line: line.length)
df.loc[df.uid == 405278, 'geometry'] = line_405278

In [25]:
uid_409283 = df.loc[df.uid == 409283, 'geometry'].iloc[0]
line_409283 = max(uid_409283.geoms, key=lambda line: line.length)
df.loc[df.uid == 409283, 'geometry'] = line_409283

In [26]:
uid_407965 = df.loc[df.uid == 407965, 'geometry'].iloc[0]
line_407965 = max(uid_407965.geoms, key=lambda line: line.length)
df.loc[df.uid == 407965, 'geometry'] = line_407965

In [27]:
uid_430436 = df.loc[df.uid == 430436, 'geometry'].iloc[0]
line_430436 = max(uid_430436.geoms, key=lambda line: line.length)
df.loc[df.uid == 430436, 'geometry'] = line_430436

In [28]:
uid_434812 = df.loc[df.uid == 434812, 'geometry'].iloc[0]
line_434812 = max(uid_434812.geoms, key=lambda line: line.length)
df.loc[df.uid == 434812, 'geometry'] = line_434812

In [29]:
uid_446771 = df.loc[df.uid == 446771, 'geometry'].iloc[0]
line_446771 = max(uid_446771.geoms, key=lambda line: line.length)
df.loc[df.uid == 446771, 'geometry'] = line_446771

In [30]:
uid_463826 = df.loc[df.uid == 463826, 'geometry'].iloc[0]
line_463826 = max(uid_463826.geoms, key=lambda line: line.length)
df.loc[df.uid == 463826, 'geometry'] = line_463826

In [31]:
uid_464338 = df.loc[df.uid == 464338, 'geometry'].iloc[0]
line_464338 = max(uid_464338.geoms, key=lambda line: line.length)
df.loc[df.uid == 464338, 'geometry'] = line_464338

In [32]:
uid_488388 = df.loc[df.uid == 488388, 'geometry'].iloc[0]
line_488388 = max(uid_488388.geoms, key=lambda line: line.length)
df.loc[df.uid == 488388, 'geometry'] = line_488388

In [33]:
uid_494950 = df.loc[df.uid == 494950, 'geometry'].iloc[0]
line_494950 = max(uid_494950.geoms, key=lambda line: line.length)
df.loc[df.uid == 494950, 'geometry'] = line_494950

In [34]:
uid_518739 = df.loc[df.uid == 518739, 'geometry'].iloc[0]
line_518739 = max(uid_518739.geoms, key=lambda line: line.length)
df.loc[df.uid == 518739, 'geometry'] = line_518739

In [35]:
uid_543273 = df.loc[df.uid == 543273, 'geometry'].iloc[0]
line_543273 = max(uid_543273.geoms, key=lambda line: line.length)
df.loc[df.uid == 543273, 'geometry'] = line_543273

In [36]:
uid_598084 = df.loc[df.uid == 598084, 'geometry'].iloc[0]
line_598084 = max(uid_598084.geoms, key=lambda line: line.length)
df.loc[df.uid == 598084, 'geometry'] = line_598084

In [37]:
uid_719148 = df.loc[df.uid == 719148, 'geometry'].iloc[0]
line_719148 = max(uid_719148.geoms, key=lambda line: line.length)
df.loc[df.uid == 719148, 'geometry'] = line_719148

In [38]:
uid_748826 = df.loc[df.uid == 748826, 'geometry'].iloc[0]
line_748826 = max(uid_748826.geoms, key=lambda line: line.length)
df.loc[df.uid == 748826, 'geometry'] = line_748826

In [39]:
uid_764931 = df.loc[df.uid == 764931, 'geometry'].iloc[0]
line_764931 = max(uid_764931.geoms, key=lambda line: line.length)
df.loc[df.uid == 764931, 'geometry'] = line_764931

In [40]:
uid_853637 = df.loc[df.uid == 853637, 'geometry'].iloc[0]
line_853637 = max(uid_853637.geoms, key=lambda line: line.length)
df.loc[df.uid == 853637, 'geometry'] = line_853637

In [41]:
uid_65002 = df.loc[df.uid == 65002, 'geometry'].iloc[0]
line_65002 = max(uid_65002.geoms, key=lambda line: line.length)
df.loc[df.uid == 65002, 'geometry'] = line_65002

rem_65002 = min(uid_65002.geoms, key=lambda line: line.length)

existing_geom_65639 = df.loc[df.uid == 65639, 'geometry'].iloc[0]
if existing_geom_65639.geom_type == "MultiLineString":
    new_geom_65639 = MultiLineString(list(existing_geom_65639.geoms) + [rem_65002])
else:
    new_geom_65639 = MultiLineString([existing_geom_65639, rem_65002])

df.loc[df.uid == 65639, 'geometry'] = ops.linemerge(new_geom_65639)

In [42]:
uid_122379 = df.loc[df.uid == 122379, 'geometry'].iloc[0]
line_122379 = max(uid_122379.geoms, key=lambda line: line.length)
df.loc[df.uid == 122379, 'geometry'] = line_122379

rem_122379 = min(uid_122379.geoms, key=lambda line: line.length)

existing_geom_122385 = df.loc[df.uid == 122385, 'geometry'].iloc[0]
if existing_geom_122385.geom_type == "MultiLineString":
    new_geom_122385 = MultiLineString(list(existing_geom_122385.geoms) + [rem_122379])
else:
    new_geom_122385 = MultiLineString([existing_geom_122385, rem_122379])

df.loc[df.uid == 122385, 'geometry'] = ops.linemerge(MultiLineString(sorted(new_geom_122385.geoms, key=lambda line: line.coords[0])))

In [43]:
uid_124389 = df.loc[df.uid == 124389, 'geometry'].iloc[0]
line_124389 = max(uid_124389.geoms, key=lambda line: line.length)
df.loc[df.uid == 124389, 'geometry'] = line_124389

rem_124389 = min(uid_124389.geoms, key=lambda line: line.length)

existing_geom_124203 = df.loc[df.uid == 124203, 'geometry'].iloc[0]
if existing_geom_124203.geom_type == "MultiLineString":
    new_geom_124203 = MultiLineString(list(existing_geom_124203.geoms) + [rem_124389])
else:
    new_geom_124203 = MultiLineString([existing_geom_124203, rem_124389])

df.loc[df.uid == 124203, 'geometry'] = ops.linemerge(MultiLineString(sorted(new_geom_124203.geoms, key=lambda line: line.coords[0])))

In [44]:
uid_285071 = df.loc[df.uid == 285071, 'geometry'].iloc[0]
line_285071 = max(uid_285071.geoms, key=lambda line: line.length)
df.loc[df.uid == 285071, 'geometry'] = line_285071

rem_285071 = min(uid_285071.geoms, key=lambda line: line.length)

existing_geom_285073 = df.loc[df.uid == 285073, 'geometry'].iloc[0]
if existing_geom_285073.geom_type == "MultiLineString":
    new_geom_285073 = MultiLineString(list(existing_geom_285073.geoms) + [rem_285071])
else:
    new_geom_285073 = MultiLineString([existing_geom_285073, rem_285071])

df.loc[df.uid == 285073, 'geometry'] = ops.linemerge(MultiLineString(sorted(new_geom_285073.geoms, key=lambda line: line.coords[0])))

In [45]:
dfm = df[df.geom_type == 'MultiLineString']
#dfm.to_csv('check.csv')
dfm.head()

,uid,RT_UNIQUE,BEGIN_MP,END_MP,lrsid,TYPE_OP,RD_NAME,CO_NAME,ADTSTATN,LASTCNT,LASTCNTYR,ADTPRIOR,ADTSINGLE,ADTCOMBO,Shape_Length,geometry,Truck_AADT,Length


## Process

In [46]:
dt = df.copy()
dt.head()

,uid,RT_UNIQUE,BEGIN_MP,END_MP,lrsid,TYPE_OP,RD_NAME,CO_NAME,ADTSTATN,LASTCNT,LASTCNTYR,ADTPRIOR,ADTSINGLE,ADTCOMBO,Shape_Length,geometry,Truck_AADT,Length
0,1180,001-CR-1001 -000,0.000,0.027,001-CR-1001 -000,2.0,TAYLOR FORD RD,Adair,0,0,0,0,0,0,125.690261,"LINESTRING (5083050.500 3562982.600, 5083024.6...",0,0.027
1,1179,001-CR-1215 -000,0.000,0.498,001-CR-1215 -000,2.0,J SAMUELL RD,Adair,0,0,0,0,0,0,2629.928650,"LINESTRING (5060128.900 3524658.500, 5060109.7...",0,0.498
2,1177,001-CR-1173 -000,0.757,0.898,001-CR-1173 -000,2.0,RUTLEDGE RD,Adair,0,0,0,0,0,0,745.439675,"LINESTRING (5092902.500 3548903.000, 5092826.0...",0,0.141
3,1176,001-CR-1070 -000,0.000,1.171,001-CR-1070 -000,2.0,REDDIE TUCKER RD,Adair,0,0,0,0,0,0,6183.250593,"LINESTRING (5078102.100 3595321.300, 5078048.8...",0,1.171
4,1167,001-CR-1001 -000,0.027,0.659,001-CR-1001 -000,2.0,TAYLOR FORD RD,Adair,0,0,0,0,0,0,3353.470541,"LINESTRING (5082945.600 3563049.400, 5082926.3...",0,0.632


In [48]:
dt.crs

<Projected CRS: EPSG:6473>
Name: NAD83(2011) / Kentucky Single Zone (ftUS)
Axis Info [cartesian]:
- X[east]: Easting (US survey foot)
- Y[north]: Northing (US survey foot)
Area of Use:
- name: United States (USA) - Kentucky.
- bounds: (-89.57, 36.49, -81.95, 39.15)
Coordinate Operation:
- name: SPCS83 Kentucky Single Zone (US survey foot)
- method: Lambert Conic Conformal (2SP)
Datum: NAD83 (National Spatial Reference System 2011)
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [49]:
dt.TYPE_OP.unique()

array([ 2.,  1., nan])

In [51]:
dt[~dt.TYPE_OP.isin([1,2])]

,uid,RT_UNIQUE,BEGIN_MP,END_MP,lrsid,TYPE_OP,RD_NAME,CO_NAME,ADTSTATN,LASTCNT,LASTCNTYR,ADTPRIOR,ADTSINGLE,ADTCOMBO,Shape_Length,geometry,Truck_AADT,Length
2232,4353,001-FD-1303 -000,0.051,0.083,001-FD-1303 -000,NaN,HOLMES BEND CAMPGROUND RD,Adair,0,0,0,0,0,0,165.046274,"LINESTRING (5062147.200 3602117.300, 5062187.7...",0,0.032
2243,4339,001-FD-1303 -010,0.051,0.083,001-FD-1303 -000,NaN,HOLMES BEND CAMPGROUND RD NC,Adair,0,0,0,0,0,0,177.029778,"LINESTRING (5062147.200 3602117.300, 5062178.7...",0,0.032
2804,7224,001-LN-9008 -010,40.896,44.130,001-LN-9008 -000,NaN,LOUIE B. NUNN CUMBERLAND EXPRESSWAY NC,Adair,85050,6013,2023,6013,659,1094,17105.233347,"LINESTRING (5013285.200 3543569.700, 5014491.9...",1753,3.234
2877,6889,001-LN-9008 -010,57.725,57.791,001-LN-9008 -000,NaN,LOUIE B. NUNN CUMBERLAND EXPRESSWAY NC,Adair,1288,7716,2022,7891,743,1224,360.582840,"LINESTRING (5095260.600 3542474.400, 5095411.4...",1967,0.066
2939,6647,001-LN-9008 -000,55.102,57.725,001-LN-9008 -000,NaN,LOUIE B. NUNN CUMBERLAND EXPRESSWAY,Adair,1288,7716,2022,7891,743,1224,13850.223606,"LINESTRING (5082518.300 3547794.100, 5082831.0...",1967,2.623
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404166,873289,120-US-0060 -000,8.072,8.130,120-US-0060 -000,NaN,US-60 BYP,Woodford,0,18695,2023,18695,788,781,353.462244,"LINESTRING (5214242.400 3912505.200, 5214580.4...",1569,0.058
404168,871523,120-KY-2113 -010,1.460,1.707,120-KY-2113 -000,NaN,FALLING SPRINGS BLVD NC,Woodford,0,4707,2023,4707,671,201,1324.117701,"LINESTRING (5209419.600 3903508.800, 5209280.0...",872,0.247
404170,871519,120-KY-2113 -010,1.707,2.041,120-KY-2113 -000,NaN,FALLING SPRINGS BLVD NC,Woodford,0,4707,2023,4707,671,201,1842.540452,"LINESTRING (5208154.400 3903899.300, 5207871.2...",872,0.334
404199,873683,120-US-0060 -010,12.097,12.237,120-US-0060 -000,NaN,LEXINGTON RD NC,Woodford,120023,44910,2023,44910,1802,2880,732.907671,"LINESTRING (5233385.300 3905751.000, 5234117.5...",4682,0.140


In [ ]:
# circle select small area for testing purposes. delete later
#dt = dt[dt.geometry.intersects(Point(5294435.58, 3905842.54).buffer(4000))]
#dt.head()

In [55]:
dt['B_Lat'], dt['B_Long'], dt['E_Lat'], dt['E_Long'] = zip(*dt['geometry'].map(latlong))

In [56]:
b_nodes = dt[['B_Lat', 'B_Long']]
b_nodes.columns = ['Lat', 'Long']

e_nodes = dt[['E_Lat', 'E_Long']]
e_nodes.columns = ['Lat', 'Long']

dt_endnodes = pd.concat([b_nodes, e_nodes], ignore_index=True) #end nodes dataframe

In [57]:
# Assign unique node id
endnodes_cnt=dt_endnodes.groupby(['Lat', 'Long']).size().reset_index(name='NodeCnt')
endnodes_cnt['NodeID'] = endnodes_cnt.index+1

# Generate the the unique node shapefile  
endnodes_unique_gpd = gpd.GeoDataFrame(endnodes_cnt, 
                                      geometry=gpd.points_from_xy(endnodes_cnt['Long'], 
                                                                 endnodes_cnt['Lat'], crs=dt.crs))

In [ ]:
#endnodes_unique_gpd.to_file('network_endnodes_unique_nodeids_sub.shp')

In [58]:
endnodes_cnt = endnodes_cnt[['Lat', 'Long', 'NodeCnt', 'NodeID']]
endnodes_cnt.columns = ['B_Lat', 'B_Long', 'B_NodeCnt', 'B_NodeID']

dt = dt.merge(endnodes_cnt, on=['B_Lat', 'B_Long'], how='left')
len(dt[pd.isnull(dt['B_NodeID'])])

0

In [59]:
endnodes_cnt.columns = ['E_Lat', 'E_Long', 'E_NodeCnt', 'E_NodeID']
dt = dt.merge(endnodes_cnt, on=['E_Lat', 'E_Long'], how='left')
len(dt[pd.isnull(dt['E_NodeID'])])

0

In [60]:
dt['TYPE_OP'] = dt['TYPE_OP'].map(lambda x: str(int(x)) if pd.notna(x) else 'D')
dt['TYPE_OP'].unique()

array(['2', '1', 'D'], dtype=object)

In [61]:
dt.head()

,uid,RT_UNIQUE,BEGIN_MP,END_MP,lrsid,TYPE_OP,RD_NAME,CO_NAME,ADTSTATN,LASTCNT,LASTCNTYR,ADTPRIOR,ADTSINGLE,ADTCOMBO,Shape_Length,geometry,Truck_AADT,Length,B_Lat,B_Long,E_Lat,E_Long,B_NodeCnt,B_NodeID,E_NodeCnt,E_NodeID
0,1180,001-CR-1001 -000,0.000,0.027,001-CR-1001 -000,2,TAYLOR FORD RD,Adair,0,0,0,0,0,0,125.690261,"LINESTRING (5083050.500 3562982.600, 5083024.6...",0,0.027,3.562983e+06,5.083050e+06,3.563049e+06,5.082946e+06,4,84972,3,85019
1,1179,001-CR-1215 -000,0.000,0.498,001-CR-1215 -000,2,J SAMUELL RD,Adair,0,0,0,0,0,0,2629.928650,"LINESTRING (5060128.900 3524658.500, 5060109.7...",0,0.498,3.524658e+06,5.060129e+06,3.524841e+06,5.057705e+06,3,64423,3,64513
2,1177,001-CR-1173 -000,0.757,0.898,001-CR-1173 -000,2,RUTLEDGE RD,Adair,0,0,0,0,0,0,745.439675,"LINESTRING (5092902.500 3548903.000, 5092826.0...",0,0.141,3.548903e+06,5.092902e+06,3.548233e+06,5.092580e+06,2,76185,1,75887
3,1176,001-CR-1070 -000,0.000,1.171,001-CR-1070 -000,2,REDDIE TUCKER RD,Adair,0,0,0,0,0,0,6183.250593,"LINESTRING (5078102.100 3595321.300, 5078048.8...",0,1.171,3.595321e+06,5.078102e+06,3.599824e+06,5.074747e+06,3,100640,1,102534
4,1167,001-CR-1001 -000,0.027,0.659,001-CR-1001 -000,2,TAYLOR FORD RD,Adair,0,0,0,0,0,0,3353.470541,"LINESTRING (5082945.600 3563049.400, 5082926.3...",0,0.632,3.563049e+06,5.082946e+06,3.563333e+06,5.079809e+06,3,85019,3,85197


In [62]:
dt['Shape_Length'] = dt['geometry'].length
dt.head()

,uid,RT_UNIQUE,BEGIN_MP,END_MP,lrsid,TYPE_OP,RD_NAME,CO_NAME,ADTSTATN,LASTCNT,LASTCNTYR,ADTPRIOR,ADTSINGLE,ADTCOMBO,Shape_Length,geometry,Truck_AADT,Length,B_Lat,B_Long,E_Lat,E_Long,B_NodeCnt,B_NodeID,E_NodeCnt,E_NodeID
0,1180,001-CR-1001 -000,0.000,0.027,001-CR-1001 -000,2,TAYLOR FORD RD,Adair,0,0,0,0,0,0,125.690261,"LINESTRING (5083050.500 3562982.600, 5083024.6...",0,0.027,3.562983e+06,5.083050e+06,3.563049e+06,5.082946e+06,4,84972,3,85019
1,1179,001-CR-1215 -000,0.000,0.498,001-CR-1215 -000,2,J SAMUELL RD,Adair,0,0,0,0,0,0,2629.928650,"LINESTRING (5060128.900 3524658.500, 5060109.7...",0,0.498,3.524658e+06,5.060129e+06,3.524841e+06,5.057705e+06,3,64423,3,64513
2,1177,001-CR-1173 -000,0.757,0.898,001-CR-1173 -000,2,RUTLEDGE RD,Adair,0,0,0,0,0,0,745.439675,"LINESTRING (5092902.500 3548903.000, 5092826.0...",0,0.141,3.548903e+06,5.092902e+06,3.548233e+06,5.092580e+06,2,76185,1,75887
3,1176,001-CR-1070 -000,0.000,1.171,001-CR-1070 -000,2,REDDIE TUCKER RD,Adair,0,0,0,0,0,0,6183.250593,"LINESTRING (5078102.100 3595321.300, 5078048.8...",0,1.171,3.595321e+06,5.078102e+06,3.599824e+06,5.074747e+06,3,100640,1,102534
4,1167,001-CR-1001 -000,0.027,0.659,001-CR-1001 -000,2,TAYLOR FORD RD,Adair,0,0,0,0,0,0,3353.470541,"LINESTRING (5082945.600 3563049.400, 5082926.3...",0,0.632,3.563049e+06,5.082946e+06,3.563333e+06,5.079809e+06,3,85019,3,85197


In [63]:
#add nc direction of two-way highways TYPE_OP == 2
dtnc = dt[dt.TYPE_OP == '2'].copy()


#reverse beginning longitude and latitude
blat = dtnc.B_Lat
blon = dtnc.B_Long
dtnc['B_Lat'] = dtnc['E_Lat']
dtnc['B_Long'] = dtnc['E_Long']
dtnc['E_Lat'] = blat
dtnc['E_Long'] = blon

#reverse bnodes and enodes
bnode = dtnc.B_NodeID
dtnc['B_NodeID'] = dtnc['E_NodeID']
dtnc['E_NodeID'] = bnode

dtnc.head()

,uid,RT_UNIQUE,BEGIN_MP,END_MP,lrsid,TYPE_OP,RD_NAME,CO_NAME,ADTSTATN,LASTCNT,LASTCNTYR,ADTPRIOR,ADTSINGLE,ADTCOMBO,Shape_Length,geometry,Truck_AADT,Length,B_Lat,B_Long,E_Lat,E_Long,B_NodeCnt,B_NodeID,E_NodeCnt,E_NodeID
0,1180,001-CR-1001 -000,0.000,0.027,001-CR-1001 -000,2,TAYLOR FORD RD,Adair,0,0,0,0,0,0,125.690261,"LINESTRING (5083050.500 3562982.600, 5083024.6...",0,0.027,3.563049e+06,5.082946e+06,3.562983e+06,5.083050e+06,4,85019,3,84972
1,1179,001-CR-1215 -000,0.000,0.498,001-CR-1215 -000,2,J SAMUELL RD,Adair,0,0,0,0,0,0,2629.928650,"LINESTRING (5060128.900 3524658.500, 5060109.7...",0,0.498,3.524841e+06,5.057705e+06,3.524658e+06,5.060129e+06,3,64513,3,64423
2,1177,001-CR-1173 -000,0.757,0.898,001-CR-1173 -000,2,RUTLEDGE RD,Adair,0,0,0,0,0,0,745.439675,"LINESTRING (5092902.500 3548903.000, 5092826.0...",0,0.141,3.548233e+06,5.092580e+06,3.548903e+06,5.092902e+06,2,75887,1,76185
3,1176,001-CR-1070 -000,0.000,1.171,001-CR-1070 -000,2,REDDIE TUCKER RD,Adair,0,0,0,0,0,0,6183.250593,"LINESTRING (5078102.100 3595321.300, 5078048.8...",0,1.171,3.599824e+06,5.074747e+06,3.595321e+06,5.078102e+06,3,102534,1,100640
4,1167,001-CR-1001 -000,0.027,0.659,001-CR-1001 -000,2,TAYLOR FORD RD,Adair,0,0,0,0,0,0,3353.470541,"LINESTRING (5082945.600 3563049.400, 5082926.3...",0,0.632,3.563333e+06,5.079809e+06,3.563049e+06,5.082946e+06,3,85197,3,85019


In [64]:
dt = pd.concat([dt, dtnc], ignore_index=True)
dt.head()

,uid,RT_UNIQUE,BEGIN_MP,END_MP,lrsid,TYPE_OP,RD_NAME,CO_NAME,ADTSTATN,LASTCNT,LASTCNTYR,ADTPRIOR,ADTSINGLE,ADTCOMBO,Shape_Length,geometry,Truck_AADT,Length,B_Lat,B_Long,E_Lat,E_Long,B_NodeCnt,B_NodeID,E_NodeCnt,E_NodeID
0,1180,001-CR-1001 -000,0.000,0.027,001-CR-1001 -000,2,TAYLOR FORD RD,Adair,0,0,0,0,0,0,125.690261,"LINESTRING (5083050.500 3562982.600, 5083024.6...",0,0.027,3.562983e+06,5.083050e+06,3.563049e+06,5.082946e+06,4,84972,3,85019
1,1179,001-CR-1215 -000,0.000,0.498,001-CR-1215 -000,2,J SAMUELL RD,Adair,0,0,0,0,0,0,2629.928650,"LINESTRING (5060128.900 3524658.500, 5060109.7...",0,0.498,3.524658e+06,5.060129e+06,3.524841e+06,5.057705e+06,3,64423,3,64513
2,1177,001-CR-1173 -000,0.757,0.898,001-CR-1173 -000,2,RUTLEDGE RD,Adair,0,0,0,0,0,0,745.439675,"LINESTRING (5092902.500 3548903.000, 5092826.0...",0,0.141,3.548903e+06,5.092902e+06,3.548233e+06,5.092580e+06,2,76185,1,75887
3,1176,001-CR-1070 -000,0.000,1.171,001-CR-1070 -000,2,REDDIE TUCKER RD,Adair,0,0,0,0,0,0,6183.250593,"LINESTRING (5078102.100 3595321.300, 5078048.8...",0,1.171,3.595321e+06,5.078102e+06,3.599824e+06,5.074747e+06,3,100640,1,102534
4,1167,001-CR-1001 -000,0.027,0.659,001-CR-1001 -000,2,TAYLOR FORD RD,Adair,0,0,0,0,0,0,3353.470541,"LINESTRING (5082945.600 3563049.400, 5082926.3...",0,0.632,3.563049e+06,5.082946e+06,3.563333e+06,5.079809e+06,3,85019,3,85197


In [65]:
# QAQC showed NC direction of Divided links followed flow direction of Cardinal direction. Reverse it to ensure proper flow
# and of through traffic and from ramps onto it.
# reverse non cardinal direction of divided highways to ensure proper flow direction

dt['dir'] = dt['RT_UNIQUE'].str[-3:] #get direction (C or NC) _o means original
dt['B_Lat_o'] = dt['B_Lat']
dt['B_Long_o'] = dt['B_Long']
dt['E_Lat_o'] = dt['E_Lat']
dt['E_Long_o'] = dt['E_Long']
dt['B_NodeCnt_o'] = dt['B_NodeCnt']
dt['B_NodeID_o'] = dt['B_NodeID']
dt['E_NodeCnt_o'] = dt['E_NodeCnt']
dt['E_NodeID_o'] = dt['E_NodeID']

In [66]:
dt['B_Lat'] = np.where(((dt['TYPE_OP'] == 'D') & (dt['dir'] == '010')), dt['E_Lat_o'], dt['B_Lat_o'])
dt['B_Long'] = np.where(((dt['TYPE_OP'] == 'D') & (dt['dir'] == '010')), dt['E_Long_o'], dt['B_Long_o'])

dt['E_Lat'] = np.where(((dt['TYPE_OP'] == 'D') & (dt['dir'] == '010')), dt['B_Lat_o'], dt['E_Lat_o'])
dt['E_Long'] = np.where(((dt['TYPE_OP'] == 'D') & (dt['dir'] == '010')), dt['B_Long_o'], dt['E_Long_o'])

dt['B_NodeCnt'] = np.where(((dt['TYPE_OP'] == 'D') & (dt['dir'] == '010')), dt['E_NodeCnt_o'], dt['B_NodeCnt_o'])
dt['B_NodeID'] = np.where(((dt['TYPE_OP'] == 'D') & (dt['dir'] == '010')), dt['E_NodeID_o'], dt['B_NodeID_o'])

dt['E_NodeCnt'] = np.where(((dt['TYPE_OP'] == 'D') & (dt['dir'] == '010')), dt['B_NodeCnt_o'], dt['E_NodeCnt_o'])
dt['E_NodeID'] = np.where(((dt['TYPE_OP'] == 'D') & (dt['dir'] == '010')), dt['B_NodeID_o'], dt['E_NodeID_o'])

dt = dt.drop(['dir','B_Lat_o','B_Long_o','E_Lat_o','E_Long_o','B_NodeCnt_o','B_NodeID_o','E_NodeCnt_o','E_NodeID_o'], axis=1)
dt.head()

,uid,RT_UNIQUE,BEGIN_MP,END_MP,lrsid,TYPE_OP,RD_NAME,CO_NAME,ADTSTATN,LASTCNT,LASTCNTYR,ADTPRIOR,ADTSINGLE,ADTCOMBO,Shape_Length,geometry,Truck_AADT,Length,B_Lat,B_Long,E_Lat,E_Long,B_NodeCnt,B_NodeID,E_NodeCnt,E_NodeID
0,1180,001-CR-1001 -000,0.000,0.027,001-CR-1001 -000,2,TAYLOR FORD RD,Adair,0,0,0,0,0,0,125.690261,"LINESTRING (5083050.500 3562982.600, 5083024.6...",0,0.027,3.562983e+06,5.083050e+06,3.563049e+06,5.082946e+06,4,84972,3,85019
1,1179,001-CR-1215 -000,0.000,0.498,001-CR-1215 -000,2,J SAMUELL RD,Adair,0,0,0,0,0,0,2629.928650,"LINESTRING (5060128.900 3524658.500, 5060109.7...",0,0.498,3.524658e+06,5.060129e+06,3.524841e+06,5.057705e+06,3,64423,3,64513
2,1177,001-CR-1173 -000,0.757,0.898,001-CR-1173 -000,2,RUTLEDGE RD,Adair,0,0,0,0,0,0,745.439675,"LINESTRING (5092902.500 3548903.000, 5092826.0...",0,0.141,3.548903e+06,5.092902e+06,3.548233e+06,5.092580e+06,2,76185,1,75887
3,1176,001-CR-1070 -000,0.000,1.171,001-CR-1070 -000,2,REDDIE TUCKER RD,Adair,0,0,0,0,0,0,6183.250593,"LINESTRING (5078102.100 3595321.300, 5078048.8...",0,1.171,3.595321e+06,5.078102e+06,3.599824e+06,5.074747e+06,3,100640,1,102534
4,1167,001-CR-1001 -000,0.027,0.659,001-CR-1001 -000,2,TAYLOR FORD RD,Adair,0,0,0,0,0,0,3353.470541,"LINESTRING (5082945.600 3563049.400, 5082926.3...",0,0.632,3.563049e+06,5.082946e+06,3.563333e+06,5.079809e+06,3,85019,3,85197


In [67]:
#get midpoint based on total line segment length for business analyst
def linemdpoint(line):
    midpoint = line.interpolate(0.5, normalized=True)
    mid_lat, mid_lon = round(midpoint.y, 6), round(midpoint.x, 6)
    return mid_lat, mid_lon

dt["Mid_Lat"], dt["Mid_Long"] = zip(*dt["geometry"].map(linemdpoint))
dt.head()

,uid,RT_UNIQUE,BEGIN_MP,END_MP,lrsid,TYPE_OP,RD_NAME,CO_NAME,ADTSTATN,LASTCNT,LASTCNTYR,ADTPRIOR,ADTSINGLE,ADTCOMBO,Shape_Length,geometry,Truck_AADT,Length,B_Lat,B_Long,E_Lat,E_Long,B_NodeCnt,B_NodeID,E_NodeCnt,E_NodeID,Mid_Lat,Mid_Long
0,1180,001-CR-1001 -000,0.000,0.027,001-CR-1001 -000,2,TAYLOR FORD RD,Adair,0,0,0,0,0,0,125.690261,"LINESTRING (5083050.500 3562982.600, 5083024.6...",0,0.027,3.562983e+06,5.083050e+06,3.563049e+06,5.082946e+06,4,84972,3,85019,3.563023e+06,5.083002e+06
1,1179,001-CR-1215 -000,0.000,0.498,001-CR-1215 -000,2,J SAMUELL RD,Adair,0,0,0,0,0,0,2629.928650,"LINESTRING (5060128.900 3524658.500, 5060109.7...",0,0.498,3.524658e+06,5.060129e+06,3.524841e+06,5.057705e+06,3,64423,3,64513,3.524638e+06,5.058857e+06
2,1177,001-CR-1173 -000,0.757,0.898,001-CR-1173 -000,2,RUTLEDGE RD,Adair,0,0,0,0,0,0,745.439675,"LINESTRING (5092902.500 3548903.000, 5092826.0...",0,0.141,3.548903e+06,5.092902e+06,3.548233e+06,5.092580e+06,2,76185,1,75887,3.548573e+06,5.092729e+06
3,1176,001-CR-1070 -000,0.000,1.171,001-CR-1070 -000,2,REDDIE TUCKER RD,Adair,0,0,0,0,0,0,6183.250593,"LINESTRING (5078102.100 3595321.300, 5078048.8...",0,1.171,3.595321e+06,5.078102e+06,3.599824e+06,5.074747e+06,3,100640,1,102534,3.597577e+06,5.076778e+06
4,1167,001-CR-1001 -000,0.027,0.659,001-CR-1001 -000,2,TAYLOR FORD RD,Adair,0,0,0,0,0,0,3353.470541,"LINESTRING (5082945.600 3563049.400, 5082926.3...",0,0.632,3.563049e+06,5.082946e+06,3.563333e+06,5.079809e+06,3,85019,3,85197,3.562774e+06,5.081329e+06


In [68]:
dt['uuid'] = dt.index+1
dt.head()

,uid,RT_UNIQUE,BEGIN_MP,END_MP,lrsid,TYPE_OP,RD_NAME,CO_NAME,ADTSTATN,LASTCNT,LASTCNTYR,ADTPRIOR,ADTSINGLE,ADTCOMBO,Shape_Length,geometry,Truck_AADT,Length,B_Lat,B_Long,E_Lat,E_Long,B_NodeCnt,B_NodeID,E_NodeCnt,E_NodeID,Mid_Lat,Mid_Long,uuid
0,1180,001-CR-1001 -000,0.000,0.027,001-CR-1001 -000,2,TAYLOR FORD RD,Adair,0,0,0,0,0,0,125.690261,"LINESTRING (5083050.500 3562982.600, 5083024.6...",0,0.027,3.562983e+06,5.083050e+06,3.563049e+06,5.082946e+06,4,84972,3,85019,3.563023e+06,5.083002e+06,1
1,1179,001-CR-1215 -000,0.000,0.498,001-CR-1215 -000,2,J SAMUELL RD,Adair,0,0,0,0,0,0,2629.928650,"LINESTRING (5060128.900 3524658.500, 5060109.7...",0,0.498,3.524658e+06,5.060129e+06,3.524841e+06,5.057705e+06,3,64423,3,64513,3.524638e+06,5.058857e+06,2
2,1177,001-CR-1173 -000,0.757,0.898,001-CR-1173 -000,2,RUTLEDGE RD,Adair,0,0,0,0,0,0,745.439675,"LINESTRING (5092902.500 3548903.000, 5092826.0...",0,0.141,3.548903e+06,5.092902e+06,3.548233e+06,5.092580e+06,2,76185,1,75887,3.548573e+06,5.092729e+06,3
3,1176,001-CR-1070 -000,0.000,1.171,001-CR-1070 -000,2,REDDIE TUCKER RD,Adair,0,0,0,0,0,0,6183.250593,"LINESTRING (5078102.100 3595321.300, 5078048.8...",0,1.171,3.595321e+06,5.078102e+06,3.599824e+06,5.074747e+06,3,100640,1,102534,3.597577e+06,5.076778e+06,4
4,1167,001-CR-1001 -000,0.027,0.659,001-CR-1001 -000,2,TAYLOR FORD RD,Adair,0,0,0,0,0,0,3353.470541,"LINESTRING (5082945.600 3563049.400, 5082926.3...",0,0.632,3.563049e+06,5.082946e+06,3.563333e+06,5.079809e+06,3,85019,3,85197,3.562774e+06,5.081329e+06,5


In [69]:
mdpoint_gpd = gpd.GeoDataFrame(dt[['uuid','RT_UNIQUE','BEGIN_MP','END_MP','LASTCNT']], 
                                      geometry=gpd.points_from_xy(dt['Mid_Long'], 
                                                                 dt['Mid_Lat'], crs=dt.crs))

print(len(mdpoint_gpd))

#mdpoint_gpd.to_file('network_midpoints.shp')

773487


In [ ]:
'''
npart = 200
total_rows = len(mdpoint_gpd)
rows_per_partition = math.ceil(total_rows / npart)

# Partition and save
for i in range(npart):
    start_idx = i * rows_per_partition
    end_idx = min((i + 1) * rows_per_partition, total_rows)
    partition = mdpoint_gpd.iloc[start_idx:end_idx]
    fc_name = f"mdpoints_partition_{i+1}"
    partition.to_file(gdb_path, layer=fc_name, ) #driver="FileGDB"
    
'''

In [70]:
nid = 226856
dt[(dt.B_NodeID == nid) | (dt.E_NodeID == nid)]#.sort_values(['B_NodeID', 'E_NodeID'])

,uid,RT_UNIQUE,BEGIN_MP,END_MP,lrsid,TYPE_OP,RD_NAME,CO_NAME,ADTSTATN,LASTCNT,LASTCNTYR,ADTPRIOR,ADTSINGLE,ADTCOMBO,Shape_Length,geometry,Truck_AADT,Length,B_Lat,B_Long,E_Lat,E_Long,B_NodeCnt,B_NodeID,E_NodeCnt,E_NodeID,Mid_Lat,Mid_Long,uuid
111471,241738,034-KY-0004 -322,0.000,0.240,034-KY-0004 -322,1,KY 4 RAMP to US 60,Fayette,0,5785,2022,6002,0,0,1264.597009,"LINESTRING (5262495.800 3906914.100, 5262576.2...",0,0.240,3.906914e+06,5.262496e+06,3.906546e+06,5.262558e+06,3,227072,3,226856,3.906950e+06,5.262969e+06,111472
113139,247122,034-US-0060 -010,4.622,4.717,034-US-0060 -000,D,VERSAILLES RD NC,Fayette,0,24846,2022,25407,843,131,501.913566,"LINESTRING (5262059.971 3906486.620, 5262065.2...",974,0.095,3.906546e+06,5.262558e+06,3.906487e+06,5.262060e+06,3,226856,2,226819,3.906517e+06,5.262309e+06,113140
113882,245132,034-US-0060 -010,4.717,4.872,034-US-0060 -000,D,VERSAILLES RD NC,Fayette,0,24846,2022,25407,843,131,821.966630,"LINESTRING (5262558.300 3906546.500, 5263374.7...",974,0.155,3.906642e+06,5.263375e+06,3.906546e+06,5.262558e+06,3,226910,3,226856,3.906594e+06,5.262966e+06,113883


In [ ]:
#dt.to_file(gdb_path, layer='AllRds_TF_processed_nodes_included')
#dt.to_file('AllRds_TF_processed_nodes_included.shp')

## Sociodemographic data

### community profile

In [ ]:
file_path = r"Z:\Eugene\Graph Neural Networks\community_profile\*.csv"
cpfiles = glob.glob(file_path)

In [ ]:
columns = ["col1", "col2", "col3", "col4", "col5", 'col6', 'col7']
all_cp_dfs = []

for file in cpfiles:
    s = pd.read_csv(file, names=columns, header=None)
    s = s.dropna(how="all").reset_index(drop=True)
    
    cpdata = pd.DataFrame({'uuid': list(set(list(map(int, s.loc[1:len(s):448, "col2"].tolist())))),})
    cpdata['pop5min'] = np.nan
    cpdata['DPop5min'] = np.nan
    cpdata['Worker5min'] = np.nan
    cpdata['Resident5min'] = np.nan
    cpdata['HH5min'] = np.nan
    cpdata['PerCapitaIncome5min'] = np.nan
    cpdata['MedianAge5min'] = np.nan
    cpdata['Employed5min'] = np.nan
    cpdata['MdHHIn5min'] = np.nan
    cpdata['Transport_MarterialMoving5min'] = np.nan
    
    curuid_index = 1
    for i in range(len(cpdata)):
        cpdata.loc[i, 'pop5min'] = s.loc[curuid_index + 8, 'col5']
        cpdata.loc[i, 'DPop5min'] = s.loc[curuid_index + 12, 'col5']
        
        cpdata.loc[i, 'Worker5min'] = s.loc[curuid_index + 13, 'col5']
        cpdata.loc[i, 'Resident5min'] = s.loc[curuid_index + 14, 'col5']
        cpdata.loc[i, 'HH5min'] = s.loc[curuid_index + 20, 'col5']
        
        cpdata.loc[i, 'MdHHIn5min'] = s.loc[curuid_index + 128, 'col5']
        cpdata.loc[i, 'PerCapitaIncome5min'] = s.loc[curuid_index + 134, 'col5']
        cpdata.loc[i, 'MedianAge5min'] = s.loc[curuid_index + 139, 'col5']
        
        cpdata.loc[i, 'Employed5min'] = s.loc[curuid_index + 312, 'col5']
        cpdata.loc[i, 'Transport_MarterialMoving5min'] = s.loc[curuid_index + 336, 'col5']
        
        curuid_index += 448  # move to next uuid in csv report
    
    all_cp_dfs.append(cpdata)

In [ ]:
community_profile = pd.concat(all_cp_dfs, ignore_index=True).drop_duplicates(['uuid']).reset_index(drop=True)
print(len(community_profile))
display(community_profile.head())
display(community_profile.tail())

In [ ]:
community_profile.to_csv('community_profile.csv', index=False)

### business profile

In [ ]:
file_path = r"Z:\Eugene\Graph Neural Networks\business_analyst\*.csv"
bpfiles = glob.glob(file_path)

In [ ]:
columns = ['col1', 'col2','col3','col4','col5','col6','col7', 'col8','col9',
           'col10','col11','col12','col13','col14',]
all_bp_dfs = []

for file in bpfiles:
    s = pd.read_csv(file, names=columns, header=None)
    s = s.dropna(how="all").reset_index(drop=True)
    
    total_business_indices = s.index[s['col1'].str.contains("Total Businesses:", na=False)].tolist()
    employees_indices = s.index[s['col1'].str.contains("Total Employees:", na=False)].tolist()

    avgmn = s.index[s['col1'].str.contains("Agriculture, Forestry, Fishing & Hunting", na=False)].tolist()
    mining = s.index[s['col1'].eq("Mining")].tolist()
    utils = s.index[s['col1'].eq("Utilities")].tolist()
    construction = s.index[s['col1'].eq("Construction")].tolist()[1::2]
    mnf = s.index[s['col1'].eq("Manufacturing")].tolist()[1::2]
    twh = s.index[s['col1'].eq("Transportation & Warehousing")].tolist()
    
    bddata = pd.DataFrame({'uuid': list(set([int(x) for x in s.col2.unique() if not pd.isna(x) and str(x).strip().isdigit()])),})
    bddata['Businesses5min'] = s.loc[total_business_indices, 'col7'].astype(int).tolist()
    bddata['Employees5min'] = s.loc[employees_indices, 'col7'].astype(int).tolist()

    bddata['AgMn5min'] = s.loc[avgmn, 'col10'].astype(int).tolist()
    bddata['Mining5min'] = s.loc[mining, 'col10'].astype(int).tolist()
    bddata['Utilities5min'] = s.loc[utils, 'col10'].astype(int).tolist()

    bddata['Construction5min'] = s.loc[construction, 'col10'].astype(int).tolist()
    bddata['Manufacturing5min'] = s.loc[mnf, 'col10'].astype(int).tolist()
    bddata['Transportation_Warehousing5min'] = s.loc[twh, 'col10'].astype(int).tolist()

    bddata['RlvntBusi5min'] = bddata[['AgMn5min', 'Mining5min','Utilities5min', 'Construction5min', 'Manufacturing5min','Transportation_Warehousing5min', ]].sum(axis=1)
    
    all_bp_dfs.append(bddata)


In [ ]:
business_profile = pd.concat(all_bp_dfs, ignore_index=True).drop_duplicates(['uuid']).reset_index(drop=True)
print(len(business_profile))
display(business_profile.head())
display(business_profile.tail())

In [ ]:
business_profile.to_csv('business_profile.csv', index=False)

### merge community and business profiles & add to data

In [ ]:
socio_data = pd.merge(community_profile, business_profile, on=['uuid'], how='inner')
print(len(community_profile))
print(len(business_profile))
print(len(socio_data))
socio_data.head()

In [ ]:
socio_data.to_csv('socioeconomic_data.csv', index=False)

In [ ]:
dt = pd.merge(dt, socio_data, on=['uuid'], how='left')
dt.head()

In [ ]:
dt.to_file(gdb_path, layer='AllRds_TF_processed_nodes_included_socioeconomic')
dt.to_file('AllRds_TF_processed_nodes_included_socioeconomic.shp')

In [ ]:
#shape area from mdpoints_tradearea files

## Speed

In [ ]:
dt = gpd.read_file('AllRds_TF_processed_nodes_included_socioeconomic.shp')
dt.head()

In [ ]:
path1 = 'Z:/PL54_HereData_2022_2023/Conflation/Conflation Table/'
conf = pd.read_csv(path1 + 'final_conflation_table_updated_newHEREmap.csv', ) #usecols = cols

conf['dir'] = conf['RT_UNIQUE'].map(lambda x: x.split('-')[-1])
conf['dir'] = np.where(conf['dir'] == '010', '000', conf['dir'])
conf['rtlrs'] = conf['RT_UNIQUE'].str[:-3] + conf['dir']

#swap where begin mp is greater than end mp
conf[['bmp','emp']] = conf[['BEGIN_MP','END_MP']].mask(conf['BEGIN_MP'] > conf['END_MP'], 
                                                               conf[['END_MP','BEGIN_MP']].values) 
display(conf.head())
display(conf.tail())

In [ ]:
dt_subcols = ['uuid', 'lrsid', 'BEGIN_MP', 'END_MP', 'TYPE_OP']

In [ ]:
conf_cols = ['rtlrs', 'bmp', 'emp', 'allrds_dir',  'h_linkdir',]

dt_conf = conf[conf_cols].merge(dt[dt_subcols], left_on='rtlrs', right_on='lrsid')
dt_conf = dt_conf[(dt_conf['bmp']<dt_conf['END_MP']) & (dt_conf['emp']>dt_conf['BEGIN_MP'])]
dt_conf['bmp_overlay'] = dt_conf[['BEGIN_MP','bmp']].max(axis=1)
dt_conf['emp_overlay'] = dt_conf[['END_MP','emp']].min(axis=1)
dt_conf['len'] = abs(dt_conf['emp_overlay']-dt_conf['bmp_overlay'])
dt_conf.rename(columns={'h_linkdir': 'LINK-DIR'}, inplace=True)
dt_conf.head(3)

In [ ]:
dt_conf[dt_conf.len == 0]

In [ ]:
speed_path = 'Z:/PL54_HereData_2022_2023/Output/'
#HERE link level ly speeds on weekdays
link_speeds = pd.read_csv(speed_path + 'HERE_Path_2023_YearlyStats.csv')
display(link_speeds.head())
display(link_speeds.tail())

In [ ]:
dt_spd = pd.merge(dt_conf, link_speeds, on=['LINK-DIR'])

dt_spd['AvgTT']=dt_spd['len']/dt_spd['AvgSpeedYear_arm']
dt_spd['StdTT']=np.where((pd.isnull(dt_spd['StdSpeedYear_arm'])) | (dt_spd['StdSpeedYear_arm']==0), 
                                        0, dt_spd['len']/dt_spd['StdSpeedYear_arm'])


dt_spd['Pcnt5TT']=dt_spd['len']/dt_spd['Pcnt5SpeedYear_arm'] #95th tt
dt_spd['Pcnt20TT']=dt_spd['len']/dt_spd['Pcnt20SpeedYear_arm'] # 80th tt
dt_spd['Pcnt50TT']=dt_spd['len']/dt_spd['Pcnt50SpeedYear_arm'] # 50th tt
dt_spd['Pcnt85TT']=dt_spd['len']/dt_spd['Pcnt85SpeedYear_arm'] # 15th tt
dt_spd.head()

In [ ]:
selcols = ['uuid', 'AvgSpeed','StdSpeed','Pcnt5Speed','Pcnt20Speed','Pcnt50Speed','Pcnt85Speed']
def speeds_agg(df, daytype):
    # Define a lambda function to compute the weighted mean based on Percent column:
    wm = lambda x: np.average(x, weights=df.loc[x.index, "len"])

    # Define a dictionary with the functions to apply for a given column:
    f = {'len': ['sum'], 'AvgTT': ['sum'], 'StdTT': ['sum'], 'Pcnt5TT': ['sum'],'Pcnt20TT': ['sum'], 
         'Pcnt50TT': ['sum'], 'Pcnt85TT': ['sum'],}

    # Groupby and aggregate with your dictionary:
    link_avg_wday= df.groupby(['uuid']).agg(f)
    link_avg_wday=link_avg_wday.reset_index()
    link_avg_wday.columns=['uuid', 'SegLength', 'SegAvgTT', 'SegStdTT', 'SegPcnt5TT' ,'SegPcnt20TT', 
                              'SegPcnt50TT', 'SegPcnt85TT',]
    link_avg_wday['AvgSpeed']=round(link_avg_wday['SegLength']/link_avg_wday['SegAvgTT'],3)
    link_avg_wday['StdSpeed']=round(link_avg_wday['SegLength']/link_avg_wday['SegStdTT'],3)
    
    link_avg_wday['Pcnt5Speed']=round(link_avg_wday['SegLength']/link_avg_wday['SegPcnt5TT'],3)
    link_avg_wday['Pcnt20Speed']=round(link_avg_wday['SegLength']/link_avg_wday['SegPcnt20TT'],3)
    link_avg_wday['Pcnt50Speed']=round(link_avg_wday['SegLength']/link_avg_wday['SegPcnt50TT'],3)
    link_avg_wday['Pcnt85Speed']=round(link_avg_wday['SegLength']/link_avg_wday['SegPcnt85TT'],3)
    
    
    
    fname='Speeds/AllRds_TF_speeds_'+ daytype +'.csv'
    link_avg_wday[selcols].to_csv(fname, index=False) #path2 +
    return link_avg_wday

In [ ]:
dt_avgspeeds = speeds_agg(dt_spd, 'weekday')

In [ ]:
dt_avgspeeds[selcols].head()

In [ ]:
dts = pd.merge(dt, dt_avgspeeds[selcols], on=['uuid'], how='left')
dt = dts.copy()
print(len(dt))
print(len(dts))
dts.head()

In [ ]:
dt.to_file(gdb_path, layer='AllRds_TF_processed_nodes_included_socioeconomic_speed')
dt.to_file('AllRds_TF_processed_nodes_included_socioeconomic_speed.shp')

## HIS Attributes

In [ ]:
his_path = 'Z:/PL54_HereData_2022_2023/Network_Screening/'

In [ ]:
his = pd.read_csv(his_path + 'TF_SubSegments_Dec172024_Data_with_TFID.csv')
his.head()

In [ ]:
dt_subcols = ['uuid', 'lrsid', 'BEGIN_MP', 'END_MP', 'TYPE_OP']

In [ ]:
'010' in dt['lrsid'].str[-3:].unique()

In [ ]:
dt_his = dt[dt_subcols].merge(his, left_on='lrsid', right_on='ROUTE_ID', how='left')
dt_his = dt_his[(dt_his['BEGIN_POINT']<dt_his['END_MP']) & (dt_his['END_POINT']>dt_his['BEGIN_MP'])]
dt_his['bmp_overlay'] = dt_his[['BEGIN_MP','BEGIN_POINT']].max(axis=1)
dt_his['emp_overlay'] = dt_his[['END_MP','END_POINT']].min(axis=1)
dt_his['length'] = abs(dt_his['emp_overlay']-dt_his['bmp_overlay'])
dt_his['Truck_Perc'] = dt_his['AVG_SINGLE_UNIT'] + dt_his['AVG_COMBINATION']
print(len(dt_his))
dt_his.head(3)

In [ ]:
dt_his['Last3']=dt_his['ROUTE_ID'].str[-3:]
dt_his['Ramp']=np.where(dt_his['Last3'].str.match('[1-9]{3}|01[1-9]'), 'Ramp', 'Mainline')

In [ ]:
dt_his['TYPE_FACILITY'].unique()

In [ ]:
dt_his['TYPE_TERRAIN'] = np.where(pd.isnull(dt_his['TYPE_TERRAIN']), 1, dt_his['TYPE_TERRAIN'])
dt_his['THROUGH_LANES']=np.where(pd.isnull(dt_his['THROUGH_LANES']), 
                                 np.where(dt_his['Ramp']=='Ramp', 1, 2), 
                                 dt_his['THROUGH_LANES'])

dt_his['LANESCRD']=np.where(pd.isnull(dt_his['LANESCRD']), 
                                 np.where(dt_his['TYPE_FACILITY']==2, dt_his['THROUGH_LANES']/2, dt_his['THROUGH_LANES']), 
                                 dt_his['LANESCRD'])
dt_his['LANESNC']=np.where(pd.isnull(dt_his['LANESNC']), 
                                 np.where(dt_his['TYPE_FACILITY']==2, dt_his['THROUGH_LANES']/2, 0), 
                                 dt_his['LANESNC'])

dt_his['LANE_WIDTH']=np.where(pd.isnull(dt_his['LANE_WIDTH']), 12, dt_his['LANE_WIDTH'])
dt_his['MEDIAN_TYPE']=np.where(pd.isnull(dt_his['MEDIAN_TYPE']), 8, dt_his['MEDIAN_TYPE'])
dt_his['MEDIAN_WIDTH']=np.where(pd.isnull(dt_his['MEDIAN_WIDTH']), 0, dt_his['MEDIAN_WIDTH'])

dt_his['SHLD_WIDTH_R']=np.where(pd.isnull(dt_his['SHLD_WIDTH_R']), 0, dt_his['SHLD_WIDTH_R'])
dt_his['SHLD_WIDTH_L']=np.where(pd.isnull(dt_his['SHLD_WIDTH_L']), 0, dt_his['SHLD_WIDTH_L'])

dt_his['ACCESS_CONTROL']=np.where(pd.isnull(dt_his['ACCESS_CONTROL']), 3, dt_his['ACCESS_CONTROL'])

In [ ]:
dt_his.uuid.is_unique

In [ ]:
# Define lambda functions
first = lambda x: x.iloc[0]
weighted_average = lambda x: round(np.average(x, weights=dt_his.loc[x.index, "length"]), 3)
round_weighted_average = lambda x: round(weighted_average(x), 0)

# most frequent value based on a given column
most_frequent_value = lambda x, column: dt_his.loc[x.index, [column, "length"]].groupby([column])['length'].sum().idxmax()


dt_his_grouped = dt_his.groupby('uuid', as_index=False).agg({
                                          
                                          'RURAL_URBAN':lambda x: most_frequent_value(x, 'RURAL_URBAN'), 
                                          'F_SYSTEM':lambda x: most_frequent_value(x, 'F_SYSTEM'),               
         
                                          'INTERCHANGES':'sum',
                                          'AT_GRADE_OTHER':'sum', 
                                          'AT_GRADE_SIGNALS':'sum', 
                                          'AT_GRADE_SIGNS':'sum', 
    
                                          'AADT': round_weighted_average,
                                          'Truck_Perc': weighted_average,
                                          'TYPE_TERRAIN':lambda x: most_frequent_value(x, 'TYPE_TERRAIN'),
                                          'THROUGH_LANES':lambda x: most_frequent_value(x, 'THROUGH_LANES'),
                                          'LANESCRD':lambda x: most_frequent_value(x, 'LANESCRD'),           
                                          'LANESNC':lambda x: most_frequent_value(x, 'LANESNC'),
                                          'LANE_WIDTH':weighted_average,
    
                                          'MEDIAN_TYPE':lambda x: most_frequent_value(x, 'MEDIAN_TYPE'),  
                                          'MEDIAN_WIDTH':round_weighted_average,  #round_weighted_average
    
                                          'SHLD_WIDTH_R':round_weighted_average,
                                          'SHLD_WIDTH_L':round_weighted_average,
                                          
                                          'ACCESS_CONTROL':lambda x: most_frequent_value(x, 'ACCESS_CONTROL'),
                                          'SPEED_LIMIT_LWA':round_weighted_average,
                                                        
                                                               
                                                                                                                
                                          })

dt_his_grouped.head(10)

In [ ]:
dt_his_grouped.describe().transpose()

In [ ]:
dt_his_grouped.to_csv('AllRds_TF_his_attributes.csv', index=False)

In [ ]:
dts = pd.merge(dt, dt_his_grouped, on='uuid', how='left')
dt = dts.copy()
dt.head()

## Betweenness Centrality

In [ ]:
bc_path = 'Z:/PL38_Criticality/March2022Rerun/compiled_output/'

In [ ]:
bc = gpd.read_file(bc_path + 'allrds_bidir_combined_zbc_mar282022.shp')
bc.rename(columns = {'RT_UNIQUE':'ROUTEID', 'BEGIN_MP':'BMP', 'END_MP':'EMP'}, inplace=True)
bc.head()

In [ ]:
dt_subcols = ['uuid', 'RT_UNIQUE', 'BEGIN_MP', 'END_MP', 'TYPE_OP']

dt_bc = dt[dt_subcols].merge(bc, left_on='RT_UNIQUE', right_on='ROUTEID', how='left')
dt_bc = dt_bc[(dt_bc['BMP']<dt_bc['END_MP']) & (dt_bc['EMP']>dt_bc['BEGIN_MP'])]
dt_bc['bmp_overlay'] = dt_bc[['BEGIN_MP','BMP']].max(axis=1)
dt_bc['emp_overlay'] = dt_bc[['END_MP','EMP']].min(axis=1)
dt_bc['length'] = abs(dt_bc['emp_overlay']-dt_bc['bmp_overlay'])
print(len(dt_bc))
dt_bc.head(3)

In [ ]:
weighted_average = lambda x: round(np.average(x, weights=dt_bc.loc[x.index, "length"]), 3)

dtbc_g = dt_bc.groupby('uuid', as_index=False).agg({'bcn_rac': weighted_average, 
                                                    'zBCn_PR_ac':weighted_average
                                                   })
dtbc_g.head()

In [ ]:
dtbc_g.to_csv('AllRds_TF_BC.csv', index=False)

In [ ]:
dts = pd.merge(dt, dtbc_g, on='uuid', how='left')
dt = dts.copy()
dt.head()

In [ ]:
dt.to_file(gdb_path, layer='AllRds_TF_processed_nodes_included_socioeconomic_speed_his_bc')
dt.to_file('AllRds_TF_processed_nodes_included_socioeconomic_speed_his_bc.shp')

## AADP

In [ ]:
dt = gpd.read_file(gdb_path, layer='AllRds_TF_processed_nodes_included_socioeconomic_speed_his_bc')
dt.head()

In [ ]:
adp = pd.read_csv('AllRds_TF_aadp.csv')
adp.head()

In [ ]:
dts = pd.merge(dt, adp, on='uuid', how='left')
dt = dts.copy()
dts.head()

In [ ]:
dt.to_file(gdb_path, layer='AllRds_TF_processed_nodes_included_socioeconomic_speed_his_bc_adp')
dt.to_file('AllRds_TF_processed_nodes_included_socioeconomic_speed_his_bc_adp.shp')

## sample for testing

In [ ]:
dt = gpd.read_file(gdb_path, layer = 'AllRds_TF_processed_nodes_included_socioeconomic_speed_his_bc_adp')
dt.head()

In [ ]:
dt.columns = ['uid', 'RT_UNIQUE', 'BEGIN_MP', 'END_MP', 'lrsid', 'TYPE_OP', 'RD_NAME',
       'CO_NAME', 'ADTSTATN', 'LASTCNT', 'LASTCNTYR', 'ADTPRIOR', 'ADTSINGLE',
       'ADTCOMBO', 'Shape_Leng', 'Truck_AADT', 'Length', 'B_Lat', 'B_Long',
       'E_Lat', 'E_Long', 'B_NodeCnt', 'B_NodeID', 'E_NodeCnt', 'E_NodeID',
       'Mid_Lat', 'Mid_Long', 'uuid', 'pop5min', 'DPop5min', 'Worker5min',
       'Resident5min', 'HH5min', 'PerCapitaIncome5min', 'MedianAge5min', 'Employed5min',
       'MdHHIn5min', 'Transport_MarterialMoving5min', 'Businesses5min', 'Employees5min', 'AgMn5min',
       'Mining5min', 'Utilities5min', 'Construction5min', 'Manufacturing5min', 'Transportation_Warehousing5min',
       'RlvntBusi5min', 'AvgSpeed', 'StdSpeed', 'Pcnt5Speed', 'Pcnt20Speed',
       'Pcnt50Speed', 'Pcnt85Speed', 'RURAL_URBAN', 'F_SYSTEM', 'INTERCHANGES',
       'AT_GRADE_OTHER', 'AT_GRADE_SIGNALS', 'AT_GRADE_SIGNS', 'AADT',
       'Truck_Perc', 'TYPE_TERRAIN', 'THROUGH_LANES', 'LANESCRD', 'LANESNC',
       'LANE_WIDTH', 'MEDIAN_TYPE', 'MEDIAN_WIDTH', 'SHLD_WIDTH_R',
       'SHLD_WIDTH_L', 'ACCESS_CONTROL', 'SPEED_LIMIT_LWA', 'bcn_rac',
       'zBCn_PR_ac', 'AADP', 'geometry', ]

In [ ]:
# circle select small area for testing purposes. delete later
dts = dt[dt.geometry.intersects(Point(5294435.58, 3905842.54).buffer(4000))]
dts.head()

In [ ]:
dts.to_file('dts.shp')

In [ ]:
endnodes_unique_gpd = gpd.read_file('network_endnodes_unique_nodeids.shp')
endnodes_unique_gpd.head()

In [ ]:
#endnodes_unique_gpd_sub = endnodes_unique_gpd[endnodes_unique_gpd.NodeID.isin(list(dts.B_NodeID) + list(dts.E_NodeID))]
#endnodes_unique_gpd_sub.head()

In [ ]:
#endnodes_unique_gpd_sub.to_file('dts_nodes.shp')

## Create Network

In [ ]:
dt = dt.drop('AADT', axis=1)
dt.rename(columns = {'LASTCNT':'AADT'}, inplace=True)
dt.head(3)

In [ ]:
rd_graph = nx.DiGraph()


nodes_data = [
    (row['NodeID'], {'lat': row['Lat'], 'long': row['Long']})
    for _, row in endnodes_unique_gpd.iterrows()
]

#add nodes
rd_graph.add_nodes_from(nodes_data)

for _, row in dt.iterrows():
    rd_graph.add_edge(row['B_NodeID'], 
                      row['E_NodeID'],
                      length=row['Length'],
                      uuid = row['uuid'],
                      mid_lat = row['Mid_Lat'],
                      mid_long = row['Mid_Long'],
                      county = row.get('CO_NAME', None),
                      typeop = row.get('TYPE_OP', None),
                      aadt = row.get('LASTCNT', None),
                      truckaadt = row.get('Truck_AADT', None),
                     )



In [ ]:
with open("motorable_ky_aadt_graph.pkl", "wb") as f:
    pickle.dump(rd_graph, f)

In [ ]:
#plt.subplots(1, 1, figsize=(12, 8), )
#pos = {node: (data['long'], data['lat']) for node, data in rd_graph.nodes(data=True)}
#nx.draw(rd_graph, pos, node_size=2, node_color='black')

In [ ]:
adj_matrix = nx.to_pandas_adjacency(rd_graph)
#adj_matrix.to_csv('adjacency_matric.csv')
adj_matrix.head()

In [ ]:
ncheck = 225825
rd_graph.in_degree(ncheck) + rd_graph.out_degree(ncheck)

In [ ]:
# Create a pyvis network
net = Network(notebook=True, height="100vh", width="100%", directed=True, 
              neighborhood_highlight=True, font_color='blue')  # Set directed=True for arrows

# Add nodes with labels
for node, data in rd_graph.nodes(data=True):
    net.add_node(node, label=str(node), color="black", font={"size": 20},  **data)  # Add node with label

# Add edges with arrows
for edge in rd_graph.edges(data=True):
    net.add_edge(edge[0], edge[1], arrows='to', **edge[2])  # Add edge with arrows

# Customize the visualization (optional)
net.toggle_physics(True)  # Enable physics for interactive behavior
#net.show_buttons(filter_=['physics'])  # Add configuration buttons

# Save the graph as an interactive HTML file
net.show("road_graph_pyvis2.html")

## Convert to Dual Graph

In [ ]:
datacols = ['TYPE_OP','AADT', 'Truck_AADT', 'Length','Mid_Lat', 'Mid_Long', 
'pop5min', 'DPop5min', 'Worker5min', 'Resident5min', 'HH5min', 'PerCapitaIncome5min', 
    'MedianAge5min', 'Employed5min', 'MdHHIn5min', 'Transport_MarterialMoving5min', 'Businesses5min', 
    'Employees5min', 'AgMn5min', 'Mining5min', 'Utilities5min', 'Construction5min', 'Manufacturing5min', 
    'Transportation_Warehousing5min', 'RlvntBusi5min',
'AvgSpeed', 'StdSpeed', 'Pcnt5Speed', 'Pcnt20Speed', 
'Pcnt50Speed', 'Pcnt85Speed', 'RURAL_URBAN', 'F_SYSTEM', 'INTERCHANGES', 'AT_GRADE_OTHER', 
'AT_GRADE_SIGNALS', 'AT_GRADE_SIGNS', 'Truck_Perc', 'TYPE_TERRAIN', 'THROUGH_LANES',
'LANESCRD','LANESNC', 'LANE_WIDTH', 'MEDIAN_TYPE', 'MEDIAN_WIDTH', 'SHLD_WIDTH_R',
'SHLD_WIDTH_L', 'ACCESS_CONTROL', 'SPEED_LIMIT_LWA', 'bcn_rac','zBCn_PR_ac', 'AADP']

In [ ]:
nid = 225430
dt_sub = dt.loc[(dt.B_NodeID == nid) | (dt.E_NodeID == nid),].sort_values(['uid'])
dt_sub

In [ ]:
sc_cols = [
    'pop5min', 'DPop5min', 'Worker5min', 'Resident5min', 'HH5min', 'PerCapitaIncome5min', 
    'MedianAge5min', 'Employed5min', 'MdHHIn5min', 'Transport_MarterialMoving5min', 'Businesses5min', 
    'Employees5min', 'AgMn5min', 'Mining5min', 'Utilities5min', 'Construction5min', 'Manufacturing5min', 
    'Transportation_Warehousing5min', 'RlvntBusi5min'
]

dt[sc_cols] = dt[sc_cols].apply(pd.to_numeric, errors='coerce')
dt.head()

In [ ]:
dt[datacols] = dt[datacols].fillna(0)

In [ ]:
print(len(dt))
dt.head()

In [ ]:
dt['RURAL_URBAN'] = np.where(dt['RURAL_URBAN'] == 0, 'Rural', dt['RURAL_URBAN'])
dt['F_SYSTEM'] = np.where(dt['F_SYSTEM'] == 0, 7, dt['F_SYSTEM'])

dt['TYPE_TERRAIN'] = np.where(dt['TYPE_TERRAIN'] == 0, 1, dt['TYPE_TERRAIN'])
dt['THROUGH_LANES'] = np.where(dt['THROUGH_LANES'] == 0, 2, dt['THROUGH_LANES'])

dt['LANE_WIDTH'] = np.where(dt['LANE_WIDTH'] == 0, 12, dt['LANE_WIDTH'])
dt['MEDIAN_TYPE'] = np.where(dt['MEDIAN_TYPE'] == 0, 8, dt['MEDIAN_TYPE'])

dt['ACCESS_CONTROL'] = np.where(dt['ACCESS_CONTROL'] == 0, 3, dt['ACCESS_CONTROL'])
dt['SPEED_LIMIT_LWA'] = np.where(dt['SPEED_LIMIT_LWA'] == 0, 55, dt['SPEED_LIMIT_LWA'])
dt.head()

In [ ]:
rd_dual_graph = nx.DiGraph()

# Extract unique uids and their data attributes
node_attrs = dt.drop_duplicates('uid').set_index('uid')[datacols].to_dict('index')

# Add nodes with attributes
rd_dual_graph.add_nodes_from(node_attrs.keys())
nx.set_node_attributes(rd_dual_graph, node_attrs)

# Build edges based on shared NodeIDs for each uid (for direction)
# Map each NodeID to the uids that reference it
nodeid_to_uids = {}
for _, row in dt.iterrows():
    nodeid_to_uids.setdefault(row['B_NodeID'], set()).add(row['uid'])
    nodeid_to_uids.setdefault(row['E_NodeID'], set()).add(row['uid'])

# For each NodeID, connect uids if B_NodeID/E_NodeID conditions are met
for nodeid, uids in nodeid_to_uids.items():
    if len(uids) > 1:  # Only if NodeID is shared
        uids = list(uids)
        for i in range(len(uids)):
            for j in range(i+1, len(uids)):
                uid1, uid2 = uids[i], uids[j]
                # Check if uid1 -> uid2 (uid1's E_NodeID = uid2's B_NodeID)
                if (dt[(dt['uid'] == uid1) & (dt['E_NodeID'] == nodeid)].any().any() and 
                    dt[(dt['uid'] == uid2) & (dt['B_NodeID'] == nodeid)].any().any()):
                    rd_dual_graph.add_edge(uid1, uid2)
                # Check reverse (uid2 -> uid1)
                if (dt[(dt['uid'] == uid2) & (dt['E_NodeID'] == nodeid)].any().any() and 
                    dt[(dt['uid'] == uid1) & (dt['B_NodeID'] == nodeid)].any().any()):
                    rd_dual_graph.add_edge(uid2, uid1)

In [ ]:
with open("motorable_ky_aadt_dual_graph2.pkl", "wb") as f:
    pickle.dump(rd_dual_graph, f)

In [ ]:
# Create a pyvis network
netd = Network(notebook=True, height="100vh", width="100%", directed=True, 
              neighborhood_highlight=True, font_color='blue')  # Set directed=True for arrows

# Add nodes with labels
for node, data in rd_dual_graph.nodes(data=True):
    netd.add_node(node, label=str(node), color="black", font={"size": 20},  **data)  # Add node with label

# Add edges with arrows
for edge in rd_dual_graph.edges(data=True):
    netd.add_edge(edge[0], edge[1], arrows='to', **edge[2])  # Add edge with arrows

# Customize the visualization (optional)
netd.toggle_physics(True)  # Enable physics for interactive behavior
#net.show_buttons(filter_=['physics'])  # Add configuration buttons

# Save the graph as an interactive HTML file
netd.show("road_dual_graph_pyvis.html")

## Read Dual Graph

In [ ]:
with open("motorable_ky_aadt_dual_graph2.pkl", "rb") as f:
    rd_dual_graph = pickle.load(f)

In [ ]:
adj_matrix = nx.to_pandas_adjacency(rd_dual_graph)
#adj_matrix.to_csv('adjacency_matrix_dual.csv')
adj_matrix.head()

In [ ]:
#adjacency matrix implementation for now is simple (0,1 connection); will see about other ways later 

In [ ]:
nodeslist = list(rd_dual_graph.nodes)
if all(isinstance(n, int) for n in nodeslist):
    sorted_nodes = sorted(nodeslist)
    is_contiguous = sorted_nodes == list(range(len(nodeslist)))
    print("Are node IDs contiguous from 0 to N-1?:", is_contiguous) #required by pytorch geometric
else:
    print("Not all node IDs are integers — mapping needed.")

In [ ]:
original_nodes = sorted(list(rd_dual_graph.nodes))
uid_to_index = {uid: i for i, uid in enumerate(original_nodes)}
index_to_uid = {i: uid for uid, i in uid_to_index.items()}

In [ ]:
nx_dgraph_indexed = nx.DiGraph()

for uid, attr in rd_dual_graph.nodes(data=True):
    nx_dgraph_indexed.add_node(uid_to_index[uid], **attr)

for u, v, edge_attrs in rd_dual_graph.edges(data=True):
        nx_dgraph_indexed.add_edge(uid_to_index[u], uid_to_index[v], **edge_attrs)

In [ ]:
print(uid_to_index[2739])
print(uid_to_index[2474])
print(uid_to_index[2469])

In [ ]:
list(rd_dual_graph.neighbors(2739))

## PyG Data Object

In [ ]:
from torch_geometric.data import Data
from torch_geometric.utils import degree, coalesce

In [ ]:
edge_list = list(nx_dgraph_indexed.edges())
edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

In [ ]:
attr_keys = list(next(iter(nx_dgraph_indexed.nodes(data=True)))[1].keys())
node_attr_lists = {key: [] for key in attr_keys}

for node_id in range(len(nx_dgraph_indexed.nodes)):
    for key in attr_keys:
        node_attr_lists[key].append(nx_dgraph_indexed.nodes[node_id].get(key, None))

In [ ]:
pyg_data = Data(edge_index=edge_index)
for key, val_list in node_attr_lists.items():
        pyg_data[key] = val_list

In [ ]:
def remove_isolated_nodes(data: Data):
    # Step 1: Identify isolated nodes
    src, dst = data.edge_index
    deg_src = degree(src, data.num_nodes)
    deg_dst = degree(dst, data.num_nodes)
    connected_mask = (deg_src + deg_dst) > 0  # keep only connected nodes

    # Step 2: Build mapping from old index → new index
    old_to_new = {}
    new_index = 0
    for old_idx, keep in enumerate(connected_mask):
        if keep:
            old_to_new[old_idx] = new_index
            new_index += 1

    # Step 3: Filter node attributes (stored as lists)
    new_data = Data()
    for key, val in data.items():
        if key == 'edge_index':
            continue  # handled below
        if isinstance(val, list) and len(val) == data.num_nodes:
            new_data[key] = [val[i] for i in range(data.num_nodes) if connected_mask[i]]
        else:
            new_data[key] = val  # keep as-is

    # Step 4: Filter edges and remap node indices
    new_edges = []
    for u, v in data.edge_index.t().tolist():
        if connected_mask[u] and connected_mask[v]:
            new_edges.append((old_to_new[u], old_to_new[v]))

    new_data.edge_index = torch.tensor(new_edges, dtype=torch.long).t().contiguous()

    return new_data

In [ ]:
pyg_data = remove_isolated_nodes(pyg_data)

In [ ]:
print(pyg_data)
pyg_data.edge_index = coalesce(pyg_data.edge_index, num_nodes=pyg_data.num_nodes)

In [ ]:
def get_isolated_nodes_directed(data):
    # Compute in-degree and out-degree
    row, col = data.edge_index
    out_deg = degree(row, num_nodes=data.num_nodes)
    in_deg = degree(col, num_nodes=data.num_nodes)
    
    # Find nodes with both in-degree and out-degree = 0
    isolated_nodes = torch.where((out_deg == 0) & (in_deg == 0))[0]
    
    return isolated_nodes

In [ ]:
#confirm no isolated nodes exist
isolated = get_isolated_nodes_directed(pyg_data)
print(len(isolated))
isolated

In [ ]:
if pyg_data.validate():
    print("pyg_data object is properly defined")
print(f'Num node features: {pyg_data.num_node_features}')
print(f'Num edge features: {pyg_data.num_edge_features}')
print(f'Edge atributes: {pyg_data.edge_attrs()}')
print(f'Num node types (should be one; all nodes represent roadway links): {pyg_data.num_node_types}')
print(f'Does Graph contains isolated nodes?: {pyg_data.has_isolated_nodes()}')
print(f'Does Graph contains self loops?: {pyg_data.has_self_loops()}')
print(f'Graph directed?: {pyg_data.is_directed()}')
print(f'Graph edges coalesced: {pyg_data.is_coalesced()}')

In [ ]:
pyg_data['Type_Operation'] = [
    'one-way highway' if val == '1' else
    'two-way highway' if val == '2' else
    'divided by median highway' if val == 'D' else val
    for val in pyg_data['TYPE_OP']
]

terrain_mapping = {
    1: 'Flat',
    2: 'Rolling',
    3: 'Mountainous'
}
pyg_data['Terrain_Type'] = [terrain_mapping.get(val, val) for val in pyg_data['TYPE_TERRAIN']]

median_mapping = {
    1: 'Concrete Barrier',
    2: 'Guardrail Barrier',
    3: 'Other Positive Barrier',
    4: 'Raised Non Mountable',
    5: 'Raised Mountable',
    6: 'Flush',
    7: 'Depressed',
    8: 'No Median'
}

pyg_data['MedianType'] = [median_mapping.get(val, val) for val in pyg_data['MEDIAN_TYPE']]

access_mapping = {
    1: 'Full',
    2: 'Partial',
    3: 'By Permit'
}

pyg_data['AccessControl'] = [access_mapping.get(val, val) for val in pyg_data['ACCESS_CONTROL']]

In [ ]:
continuous_features = ['AADP', 'THROUGH_LANES', 'LANE_WIDTH','SHLD_WIDTH_R', 'SPEED_LIMIT_LWA', 
                       'F_SYSTEM', 'MdHHIn5min', 'DPop5min', 'HH5min', 'Businesses5min',
                       'Employed5min', 'AvgSpeed', 'Pcnt85Speed']

In [ ]:
pyg_data.x = torch.stack([
    torch.tensor(pyg_data[col], dtype=torch.float) 
    for col in continuous_features
], dim=1)
print(pyg_data.x.size())

In [ ]:
pyg_data.y = torch.tensor(pyg_data['AADT'], dtype=torch.float).unsqueeze(1)
print(pyg_data.y.size())

In [ ]:
y_nonzero = pyg_data.y[pyg_data.y > 0].squeeze()

# Sort for percentile lookup
y_sorted, _ = torch.sort(y_nonzero)

n = y_sorted.numel()
percentile = lambda p: y_sorted[int(p * n)]

describe_dict = {
    'count': y_nonzero.numel(),
    'mean': y_nonzero.mean().item(),
    'std': y_nonzero.std(unbiased=False).item(),  # match pandas (ddof=0)
    'min': y_sorted[0].item(),
    '1%': percentile(0.01).item(),
    '2.5%': percentile(0.025).item(),
    '5%': percentile(0.05).item(),
    '25%': percentile(0.25).item(),
    '50%': y_nonzero.median().item(),
    '75%': percentile(0.75).item(),
    '95%': percentile(0.95).item(),
    '97.5%': percentile(0.975).item(),
    '99%': percentile(0.99).item(),
    'max': y_sorted[-1].item()
}

describe_aadt = pd.DataFrame(describe_dict, index=['AADT'])
describe_aadt

In [ ]:
pcnt_value = describe_dict['1%']
num_below = (y_nonzero < pcnt_value).sum().item()
print(f"Number of nodes with AADT < 1th percentile ({pcnt_value:,.2f}): {num_below}")

In [ ]:
p99_value = describe_dict['99%']
num_above_99 = (y_nonzero > p99_value).sum().item()
print(f"Number of nodes with AADT > 99th percentile ({p99_value:,.2f}): {num_above_99}")

### Prep data

In [ ]:
from sklearn.preprocessing import StandardScaler

# Convert to numpy
X_np = pyg_data.x.numpy()

# Fit and transform 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_np)

# Replace in graph
pyg_data.x = torch.tensor(X_scaled, dtype=torch.float)
print(pyg_data.x.size())

In [ ]:
cat_cols  = ['Type_Operation','RURAL_URBAN','Terrain_Type','MedianType','AccessControl']

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    pyg_data[col+'_encoded'] = le.fit_transform(pyg_data[col])
    encoders[col] = le  # Save for inverse transform later if needed

In [ ]:
print(pyg_data)

In [ ]:
print(encoders)
encoders['Type_Operation']

In [ ]:
encoders['Type_Operation'].classes_

In [ ]:
cat_dims = [len(encoders[col].classes_) for col in cat_cols]  # input to each Embedding()
emb_dims = [min(50, (n + 1) // 2) for n in cat_dims]           # common embedding rule of thumb

In [ ]:
emb_dims

In [ ]:
from sklearn.model_selection import train_test_split

# nodes with measured AADT
has_aadt_mask = pyg_data.y.squeeze() > 0
y_nonzero = pyg_data.y.squeeze()[has_aadt_mask]

# 1st percentile cutoff
percentile_1 = torch.quantile(y_nonzero, 0.01) #1th percentile

# filter labeled_indices using 1st percentile threshold
labeled_indices = (pyg_data.y.squeeze() > percentile_1).nonzero(as_tuple=True)[0]

# Train/Val/Test split from labeled nodes only (no outliers)
train_idx, temp_idx = train_test_split(labeled_indices, test_size=0.3, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

# boolean masks (all nodes) - initializing
num_nodes = pyg_data.num_nodes
pyg_data.train_mask = torch.zeros(num_nodes, dtype=torch.bool)
pyg_data.val_mask = torch.zeros(num_nodes, dtype=torch.bool)
pyg_data.test_mask = torch.zeros(num_nodes, dtype=torch.bool)

# train/validation/test masks
pyg_data.train_mask[train_idx] = True
pyg_data.val_mask[val_idx] = True
pyg_data.test_mask[test_idx] = True


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, summary

In [ ]:
def mean_absolute_percentage_error(y_true, y_pred):
    # Avoid division by zero
    epsilon = 1e-10
    return torch.mean(torch.abs((y_true - y_pred) / (y_true + epsilon))) * 100

## Model Architectures

### GCNConv

In [ ]:
class GNNModel1(nn.Module):
    def __init__(self, continuous_dims, gnn_in_dims, gnn_out_dims,):
        super().__init__()
        
        # Embedding layers with rule-of-thumb dimensions
        self.embeddings = nn.ModuleDict({
            'Type_Operation_encoded': nn.Embedding(3, 3),
            'RURAL_URBAN_encoded': nn.Embedding(2, 2),
            'Terrain_Type_encoded': nn.Embedding(3, 3),
            'MedianType_encoded': nn.Embedding(8, 5),
            'AccessControl_encoded': nn.Embedding(3, 3),
        })
        
        total_embed_dim = sum(emb.embedding_dim for emb in self.embeddings.values())
        input_dim = total_embed_dim + continuous_dims
        
        # GNN layers
        self.conv1 = GCNConv(input_dim, gnn_in_dims)
        self.conv2 = GCNConv(gnn_in_dims, gnn_out_dims)
        
        # MLP head
        self.regressor = nn.Sequential(
            nn.Linear(gnn_out_dims, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
        
        
    def forward(self, data):
        # Embedding categorical features
        embedded = []
        for col in self.embeddings:
            x_cat = data[col].long()
            emb = self.embeddings[col](x_cat)
            embedded.append(emb)
        x_cat_embed = torch.cat(embedded, dim=1)
        
        # continuous features
        x_cont = data.x
        
        # concat continuous and embedded categorical feature
        x = torch.cat([x_cat_embed, x_cont], dim=1)
        
        # GNN forward
        x = F.relu(self.conv1(x, data.edge_index))
        x = F.relu(self.conv2(x, data.edge_index))
        outpred = self.regressor(x)
        
        return outpred

In [ ]:
encoded_categorical_cols = [
    'Type_Operation_encoded',
    'RURAL_URBAN_encoded',
    'Terrain_Type_encoded',
    'MedianType_encoded',
    'AccessControl_encoded'
]

for col in encoded_categorical_cols:
    if isinstance(pyg_data[col], np.ndarray):
        pyg_data[col] = torch.tensor(pyg_data[col], dtype=torch.long)
    elif isinstance(pyg_data[col], list):
        pyg_data[col] = torch.tensor(pyg_data[col], dtype=torch.long)

In [ ]:
import copy

def train_and_evaluate(data, model, epochs=100, lr=1e-3, save_path='best_model_checkpoint.pt'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Training device: {device}")

    model = model.to(device)
    data = data.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    best_val_rmse = float('inf')
    best_model_state = None

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        out = model(data)
        loss = loss_fn(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        # Validation
        model.eval()
        with torch.no_grad():
            val_pred = out[data.val_mask]
            test_pred = out[data.test_mask]
            train_pred = out[data.train_mask]

            val_true = data.y[data.val_mask]
            test_true = data.y[data.test_mask]
            train_true = data.y[data.train_mask]

            val_loss = loss_fn(val_pred, val_true)
            test_loss = loss_fn(test_pred, test_true)

            # RMSE
            train_rmse = torch.sqrt(loss).item()
            val_rmse = torch.sqrt(val_loss).item()
            test_rmse = torch.sqrt(test_loss).item()

            # MAPE
            train_mape = mean_absolute_percentage_error(train_true, train_pred).item()
            val_mape = mean_absolute_percentage_error(val_true, val_pred).item()
            test_mape = mean_absolute_percentage_error(test_true, test_pred).item()

        # Save best model
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            mape_at_best_val_rmse = val_mape
            best_model_state = copy.deepcopy(model.state_dict())
            torch.save(best_model_state, save_path)

        # Progress print
        if (epoch + 1) % 50 == 0:
            print(f"\nProgress @ Epoch {epoch+1}:")
            print(f"  Best Val RMSE so far: {best_val_rmse:.4f}")
            print(f" MAPE @ Best Validation Checkpoint: {mape_at_best_val_rmse:.2f}")
            print(f"  Val RMSE:             {val_rmse:.4f}   | Val MAPE:  {val_mape:.2f}%")
            print(f"  Test RMSE:            {test_rmse:.4f}  | Test MAPE: {test_mape:.2f}%\n")

    print(f"\nBest model saved with val RMSE: {best_val_rmse:.4f} to '{save_path}'")

In [ ]:
continuous_dims = pyg_data.x.size(1)
model = GNNModel1(continuous_dims=continuous_dims, gnn_in_dims=64, gnn_out_dims=32)

train_and_evaluate(pyg_data, model, epochs=20000, lr=1e-3)

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.load_state_dict(torch.load('best_model_checkpoint.pt', map_location=device))
model.to(device)
model.eval()

In [ ]:
# Inference
with torch.no_grad():
    out = model(pyg_data.to(device))
    y_true = pyg_data.y.squeeze()
    y_pred = out.squeeze()

# Move to CPU 
y_true = y_true.cpu()
y_pred = y_pred.cpu()

# Filter non-zero AADT
mask_nonzero = y_true > 0
y_true_nz = y_true[mask_nonzero]
y_pred_nz = y_pred[mask_nonzero]

# Define AADT thresholds
thresholds = [500, 2000, 5_000, 10_000, 20000, 35000, 55000, 85000, 125000,250000]
results = []

lower = 0
for upper in thresholds:
    mask = (y_true_nz >= lower) & (y_true_nz < upper)
    if mask.sum() == 0:
        results.append((f"{lower}-{int(upper)}", None, None))
    else:
        y_t = y_true_nz[mask].numpy()
        y_p = y_pred_nz[mask].numpy()

        rmse = mean_squared_error(y_t, y_p, squared=False)
        mape = mean_absolute_percentage_error(y_t, y_p) * 100

        results.append((f"{lower}-{int(upper)}", rmse, mape))
    
    lower = upper


print("AADT Range       | RMSE        | MAPE (%)")
print("-----------------|-------------|-----------")
for rng, rmse, mape in results:
    if rmse is None:
        print(f"{rng:<15} | No samples")
    else:
        print(f"{rng:<15} | {rmse:>10.2f} | {mape:>8.2f}")


In [ ]:
df_eval = pd.DataFrame(results, columns=["AADT_Range", "RMSE", "MAPE"])
df_eval

In [ ]:
from sparsemax import Sparsemax

class GLU(nn.Module):
    """Gated Linear Unit (used in TabNet)"""
    def __init__(self, dim=-1):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        A, B = x.chunk(2, dim=self.dim)
        return A * torch.sigmoid(B)

class TabNetAttention(nn.Module):
    """TabNet-style attention with GLU and Sparsemax"""
    def __init__(self, in_dim):
        super().__init__()
        self.attention_net = nn.Sequential(
            nn.Linear(in_dim, in_dim * 2),  # Double dim for GLU
            GLU(dim=-1),
            nn.LayerNorm(in_dim),           # Stabilize training
            nn.Linear(in_dim, in_dim),
            Sparsemax(dim=-1)               # Enforce sparsity
        )
        # Initialize last layer to near-zero
        nn.init.xavier_normal_(self.attention_net[-2].weight, gain=0.01)
        nn.init.zeros_(self.attention_net[-2].bias)

    def forward(self, x):
        mask = self.attention_net(x)
        return x * mask  # Feature selection

class GNNWithTabNetHead(nn.Module):
    def __init__(self, continuous_dims, gnn_in_dims, gnn_out_dims):
        super().__init__()

        # Embedding layers
        self.embeddings = nn.ModuleDict({
            'Type_Operation_encoded': nn.Embedding(3, 3),
            'RURAL_URBAN_encoded': nn.Embedding(2, 2),
            'Terrain_Type_encoded': nn.Embedding(3, 3),
            'MedianType_encoded': nn.Embedding(8, 5),
            'AccessControl_encoded': nn.Embedding(3, 3),
        })

        total_embed_dim = sum(emb.embedding_dim for emb in self.embeddings.values())
        input_dim = total_embed_dim + continuous_dims

        # GCN layers
        self.conv1 = GCNConv(input_dim, gnn_in_dims, add_self_loops=True)
        self.conv2 = GCNConv(gnn_in_dims, gnn_out_dims, add_self_loops=True)

        # TabNet-style attention (with GLU+Sparsemax)
        self.attention = TabNetAttention(gnn_out_dims)

        # Regressor
        self.regressor = nn.Sequential(
            nn.Linear(gnn_out_dims, 16),
            nn.GELU(),          # Matches TabNet's style
            nn.Dropout(0.3),
            nn.Linear(16, 1))
        
        # Track feature importance (TabNet-like)
        # Initialize feature importance as buffer to ensure proper device placement
        self.register_buffer('feature_importance', torch.zeros(gnn_out_dims))

    def forward(self, data):
        # Embed categorical features
        embedded = []
        for col in self.embeddings:
            x_cat = data[col].long()
            emb = self.embeddings[col](x_cat)
            embedded.append(emb)
        x_cat_embed = torch.cat(embedded, dim=1)

        # Concatenate with continuous features
        x_cont = data.x
        x = torch.cat([x_cat_embed, x_cont], dim=1)

        # GCN forward pass
        x = F.relu(self.conv1(x, data.edge_index))
        x = F.relu(self.conv2(x, data.edge_index))

        # TabNet-style attention
        x_attn = self.attention(x)
        self.feature_importance += x_attn.mean(0).detach()  # Track importance
        
        outpred = self.regressor(x_attn)

        return outpred